# TensorFly — Run all (A100)

Select an **A100** paid GPU, then click **Runtime → Run all**. The optional Qwen acceleration is attempted without replacing Colab's Torch installation.


In [ ]:
# SETUP — automatic
from pathlib import Path
import os, subprocess, sys

repo = Path('/content/fly-inference-optimizer')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not a TensorFly git clone. Use a clean Colab runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/MrFaruk0/fly-inference-optimizer.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pyarrow', 'accelerate', 'safetensors', 'transformers', 'einops'], check=True)
source_root = str(repo / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select an A100 paid CUDA runtime, then use Runtime → Run all again.')
name = torch.cuda.get_device_name(0)
if 'A100' not in name.upper():
    raise RuntimeError(f'This notebook is configured for A100; detected {name!r}.')

# Optional Qwen3.5 DeltaNet acceleration. Never replace Colab's torch stack.
kernel_install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'causal-conv1d', 'flash-linear-attention'], capture_output=True, text=True)
try:
    import causal_conv1d, fla
    print('A100 Qwen kernels: enabled')
except ImportError:
    print('A100 Qwen kernels unavailable; using Transformers reference kernels (correct but slower).')
    if kernel_install.stderr:
        print(kernel_install.stderr[-1000:])

import tensorfly
print('TensorFly:', tensorfly.__version__)
print('GPU:', name)


In [ ]:
# COMPLETE EXPERIMENT — automatic
import subprocess
from tensorfly import DEFAULT_PROMPTS, TensorFlyExperiment

prepared = tensorfly.prepare()
experiment = TensorFlyExperiment(model='Qwen/Qwen3.5-9B')
tensorfly_records = experiment.run(prompt_corpus=DEFAULT_PROMPTS, trials=20)
baselines = experiment.compare_baselines(prompt_corpus=DEFAULT_PROMPTS, trials=20)
replay_path = experiment.export_video()

summary = {name: {'final_reward': rows[-1].reward, 'mean_reward': sum(row.reward for row in rows) / len(rows), 'final_config': rows[-1].resulting_next_config} for name, rows in baselines.items()}
print('MaleCNS report:', prepared.dataset.report)
print('Baseline summary:', summary)
print('Replay:', replay_path)

server = subprocess.Popen([sys.executable, '-m', 'http.server', '8000', '--directory', 'viewer'])
from google.colab import output
output.serve_kernel_port_as_window(8000)
